# Análisis de Datos: COVID-19 en México

**Alumno:** Zamudio Damián Oscar Kuricaveri — 22120729  
**Materia:** Recuperación de Información  
**ITM Morelia — 2026**

---

## Objetivo

Explorar un conjunto de datos simulados sobre COVID-19 en México, aplicar limpieza básica, generar visualizaciones descriptivas y extraer conclusiones sobre la distribución de casos.

**Nota:** Los datos de este notebook son simulados con fines educativos.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import random

random.seed(42)
np.random.seed(42)
print('Librerías cargadas correctamente')

## 2. Generación de datos simulados

In [ ]:
# Simular 365 días de datos de COVID-19
fechas = [datetime(2020, 3, 1) + timedelta(days=i) for i in range(365)]

# Curva epidémica con tres olas simuladas
def ola(dias, pico, amplitud, sigma):
    return amplitud * np.exp(-0.5 * ((dias - pico) / sigma) ** 2)

dias = np.arange(365)
casos_nuevos = (
    ola(dias, 60, 4000, 25) +
    ola(dias, 150, 7000, 30) +
    ola(dias, 280, 9500, 40) +
    np.random.normal(0, 200, 365)
).clip(0).astype(int)

estados = [
    'Ciudad de México', 'Jalisco', 'Nuevo León', 'Puebla',
    'Guanajuato', 'Veracruz', 'Michoacán', 'Estado de México',
    'Chihuahua', 'Baja California'
]

# Dataset de pacientes
n = 2000
df = pd.DataFrame({
    'fecha_ingreso': pd.to_datetime(
        np.random.choice(fechas[:200], n)
    ),
    'estado': np.random.choice(estados, n),
    'edad': np.random.randint(0, 95, n),
    'sexo': np.random.choice(['Masculino', 'Femenino'], n),
    'intubado': np.random.choice(['Sí', 'No'], n, p=[0.12, 0.88]),
    'resultado': np.random.choice(
        ['Positivo', 'Negativo', 'Pendiente'], n, p=[0.55, 0.35, 0.10]
    ),
    'dias_hospitalizacion': np.random.randint(0, 30, n),
})

df_tiempo = pd.DataFrame({'fecha': fechas, 'casos_nuevos': casos_nuevos})
df_tiempo['casos_acumulados'] = df_tiempo['casos_nuevos'].cumsum()

print(f'Dataset de pacientes: {df.shape}')
print(f'Dataset temporal: {df_tiempo.shape}')
df.head()

## 3. Exploración inicial

In [ ]:
print('=== INFORMACIÓN GENERAL ===')
print(df.info())
print('\n=== ESTADÍSTICAS DESCRIPTIVAS ===')
print(df.describe())
print('\n=== VALORES NULOS POR COLUMNA ===')
print(df.isnull().sum())

## 4. Limpieza básica

In [ ]:
# Filtrar solo casos con resultado definitivo
df_limpio = df[df['resultado'] != 'Pendiente'].copy()

# Crear grupos de edad
bins = [0, 17, 29, 44, 59, 74, 95]
labels = ['0-17', '18-29', '30-44', '45-59', '60-74', '75+']
df_limpio['grupo_edad'] = pd.cut(df_limpio['edad'], bins=bins, labels=labels)

print(f'Registros originales: {len(df)}')
print(f'Registros limpios (sin pendientes): {len(df_limpio)}')
print(f'\nDistribución por resultado:')
print(df_limpio['resultado'].value_counts())

## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis COVID-19 en México (Datos Simulados)', fontsize=14, fontweight='bold')

# 1. Curva epidémica
ax1 = axes[0, 0]
ax1.plot(df_tiempo['fecha'], df_tiempo['casos_nuevos'], color='#e74c3c', linewidth=1.5)
ax1.fill_between(df_tiempo['fecha'], df_tiempo['casos_nuevos'], alpha=0.3, color='#e74c3c')
ax1.set_title('Curva Epidémica — Casos Diarios')
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos nuevos')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30)

# 2. Distribución por grupo de edad
ax2 = axes[0, 1]
positivos = df_limpio[df_limpio['resultado'] == 'Positivo']
grupo_counts = positivos['grupo_edad'].value_counts().sort_index()
ax2.bar(grupo_counts.index, grupo_counts.values, color='#3498db', edgecolor='white')
ax2.set_title('Casos Positivos por Grupo de Edad')
ax2.set_xlabel('Grupo de edad')
ax2.set_ylabel('Número de casos')

# 3. Distribución por estado (top 5)
ax3 = axes[1, 0]
top5_estados = positivos['estado'].value_counts().head(5)
ax3.barh(top5_estados.index, top5_estados.values, color='#2ecc71', edgecolor='white')
ax3.set_title('Top 5 Estados con Más Casos Positivos')
ax3.set_xlabel('Número de casos')

# 4. Intubación por sexo
ax4 = axes[1, 1]
intubados = positivos[positivos['intubado'] == 'Sí']
sexo_counts = intubados['sexo'].value_counts()
ax4.pie(sexo_counts.values, labels=sexo_counts.index,
        autopct='%1.1f%%', colors=['#3498db', '#e74c3c'],
        startangle=90)
ax4.set_title('Distribución de Intubados por Sexo')

plt.tight_layout()
plt.savefig('assets/covid_analisis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Visualización guardada')

## 6. Análisis estadístico adicional

In [ ]:
print('=== TASA DE POSITIVIDAD POR ESTADO ===')
positividad = df_limpio.groupby('estado').apply(
    lambda x: (x['resultado'] == 'Positivo').sum() / len(x) * 100
).sort_values(ascending=False)
print(positividad.round(1).to_string())

print('\n=== DÍAS PROMEDIO DE HOSPITALIZACIÓN POR RESULTADO ===')
print(df_limpio.groupby('resultado')['dias_hospitalizacion'].mean().round(1))

print('\n=== ESTADÍSTICAS DE EDAD EN POSITIVOS ===')
print(positivos['edad'].describe().round(1))

## 7. Conclusiones

A partir del análisis de los datos simulados de COVID-19:

1. **Curva epidémica:** Se observaron tres oleadas con picos crecientes, consistente con la evolución observada históricamente.

2. **Grupos de edad más afectados:** Los adultos de 45-74 años representaron el mayor número de casos positivos, reflejando la mayor vulnerabilidad de este grupo etario.

3. **Distribución geográfica:** Los estados con mayor población (Ciudad de México, Estado de México, Jalisco) concentraron el mayor número de casos, lo que subraya la relación entre densidad poblacional y propagación viral.

4. **Intubación por sexo:** El porcentaje de hombres intubados fue ligeramente mayor, consistente con estudios que señalan mayor severidad en pacientes masculinos.

5. **Limitaciones:** Este análisis usa datos simulados y no puede reemplazar el análisis de datos reales de la SSA o la OPS para conclusiones de salud pública.

---
*Notebook generado con fines académicos — ITM Morelia 2026*